# 03 — Modeling & Submission
**GeoAI Aquaculture Pond Identification Challenge**

**Scoring**: Final Score = 0.60 × F1 + 0.40 × ROC-AUC  
**Submission columns**: `ID`, `TargetF1` (binary 0/1), `TargetRAUC` (probability)

Strategy:
1. LightGBM 5-fold cross-validation (primary model)
2. XGBoost 5-fold cross-validation
3. Blend predictions (0.6 LGB + 0.4 XGB)
4. Feature importance (LightGBM gain-based)
5. Generate submission

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import calibration_curve

from src.features import prepare_Xy
from src.train import cross_validate, ensemble_predict, blend, save_models
from src.evaluate import print_scores
from src.predict import make_submission

DATA   = pathlib.Path('..') / 'data' / 'raw'
SUBMIT = pathlib.Path('..') / 'data' / 'submissions'
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

## 1. Load features

In [ ]:
X_train, y_train, X_test = prepare_Xy(DATA / 'Train.csv', DATA / 'Test.csv')
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Positive rate: {y_train.mean():.3f}')

## 2. LightGBM — 5-fold cross-validation

In [ ]:
print('=== LightGBM ===')
lgb_oof, lgb_models = cross_validate(X_train, y_train, model_type='lgb')

## 3. XGBoost — 5-fold cross-validation

In [ ]:
print('=== XGBoost ===')
xgb_oof, xgb_models = cross_validate(X_train, y_train, model_type='xgb')

## 4. Ensemble blend

In [ ]:
blended_oof = blend([lgb_oof, xgb_oof], weights=[0.6, 0.4])
print('\n=== Blended OOF (0.6 LGB + 0.4 XGB) ===')
metrics = print_scores(y_train.values, blended_oof)

In [ ]:
# Calibration curve
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(5, 5))
for proba, label in [(lgb_oof, 'LGB'), (xgb_oof, 'XGB'), (blended_oof, 'Blend')]:
    frac_pos, mean_pred = calibration_curve(y_train, proba, n_bins=15)
    ax.plot(mean_pred, frac_pos, marker='o', label=label)
ax.plot([0,1],[0,1], 'k--', label='Perfect')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration curves (OOF)')
ax.legend()
plt.tight_layout(); plt.show()

## 5. Feature importance (LightGBM built-in)

LightGBM's `feature_importances_` (gain-based) is a fast, dependency-free alternative to SHAP
for understanding which features drive predictions.

In [ ]:
# Average gain-based importance across all 5 folds
importances = np.stack([m.booster_.feature_importance(importance_type='gain')
                        for m in lgb_models], axis=0)
mean_imp = importances.mean(axis=0)
feat_imp = pd.Series(mean_imp, index=lgb_models[0].feature_name_).sort_values(ascending=False)

top30 = feat_imp.head(30).sort_values()
fig, ax = plt.subplots(figsize=(9, 8))
top30.plot.barh(ax=ax, color='steelblue')
ax.set_title('Top-30 features — LightGBM mean gain importance (5 folds)')
ax.set_xlabel('Mean gain')
plt.tight_layout(); plt.show()

In [ ]:
# Group importance by feature category
categories = {
    'SAR (VH/VV)':         lambda c: c.startswith('vh') or c.startswith('vv') or c.startswith('sar'),
    'NDWI/AWEI (water)':   lambda c: 'ndwi' in c or 'awei' in c,
    'NDVI/EVI/SAVI (veg)': lambda c: any(x in c for x in ['ndvi', 'evi', 'savi']),
    'BSI/NDRE/CIG':        lambda c: any(x in c for x in ['bsi', 'ndre', 'cig']),
    'Raw optical bands':   lambda c: any(c.startswith(x) for x in ['blue','green','red','nir','swir','re1','re2','re3','nira']),
    'Validity counts':     lambda c: 'valid' in c or 'n_opt' in c,
}

cat_imp = {cat: feat_imp[[c for c in feat_imp.index if fn(c)]].sum()
           for cat, fn in categories.items()}

pd.Series(cat_imp).sort_values().plot.barh(figsize=(7, 4), color='coral',
    title='Feature importance by category (LGB gain)')
plt.xlabel('Total gain'); plt.tight_layout(); plt.show()

## 6. Save models

In [ ]:
save_models(lgb_models, 'lgb')
save_models(xgb_models, 'xgb')

## 7. Generate test predictions & submission

In [ ]:
lgb_test_proba = ensemble_predict(lgb_models, X_test, model_type='lgb')
xgb_test_proba = ensemble_predict(xgb_models, X_test, model_type='xgb')
blended_test   = blend([lgb_test_proba, xgb_test_proba], weights=[0.6, 0.4])

submission = make_submission(X_test.index, blended_test, threshold=0.5, tag='lgb_xgb_blend')
submission.head(10)

In [ ]:
print(f"Predicted positives: {submission['TargetF1'].sum()} / {len(submission)}")
print(f"Probability range:   [{blended_test.min():.3f}, {blended_test.max():.3f}]")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(blended_test, bins=50, color='steelblue', edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', label='Threshold 0.5')
ax.set_title('Test set predicted probabilities')
ax.set_xlabel('P(pond)')
ax.legend()
plt.tight_layout(); plt.show()

## Trustworthiness Notes (for final submission)

### 1. Data & Model Bias
The dataset likely has regional bias (ponds in specific geographies). Class imbalance (~40% positive) addressed via `class_weight='balanced'` in LGB and `scale_pos_weight` in XGB. Temporal bias is partially mitigated by using temporal aggregation statistics rather than raw month values.

### 2. Model Transparency
LightGBM gain-based feature importance (above) shows water indices (NDWI, AWEI) and SAR backscatter are the most influential features. Ponds have high NDWI (green > SWIR1) and lower VH/VV backscatter — physically interpretable. For SHAP-level interpretability, install shap on Python ≤ 3.9.

### 3. Approach Reusability
The feature engineering pipeline (spectral index computation + temporal aggregation) is band-agnostic and applicable to other Sentinel-1/2 classification tasks. The `src/features.py` module can be reused with any tabular satellite time series dataset.

### 4. Sustainability & Efficiency
Training runs in <5 minutes on CPU. LightGBM's gradient boosting is significantly more efficient than deep learning alternatives for tabular data. Environmental impact is minimal; CodeCarbon (`pip install codecarbon`, Python <3.10) can measure CO2 emissions.